In [0]:
fato_acidente = spark.table("prf_acidentes.gold.fato_acidente")
dim_tempo = spark.table("prf_acidentes.gold.dim_tempo")
dim_local = spark.table("prf_acidentes.gold.dim_local")
dim_causa = spark.table("prf_acidentes.gold.dim_causa")
dim_condicao = spark.table("prf_acidentes.gold.dim_condicao")
dim_classificacao = spark.table("prf_acidentes.gold.dim_classificacao")

print("Tabelas carregadas com sucesso.")

Pergunta 1: Ranking de BRs por acidentes e gravidade

In [0]:
from pyspark.sql import functions as F

df_p1 = (
    fato_acidente
    .join(dim_local, on="id_local", how="left")
    .filter(F.col("br") != 0)
    .groupBy("br")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves"),
        F.sum("feridos_leves").alias("total_feridos_leves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .orderBy(F.desc("qtd_acidentes"))
)

display(df_p1.limit(15))

**Interpretação — Resposta à Pergunta 1**

BR-101 e BR-116 dominam disparado em volume de acidentes (37.426 e 33.303 respectivamente), muito à frente da 3ª colocada (BR-381, 10.252). Isso é coerente com a realidade: são as duas rodovias federais mais extensas e movimentadas do país, cortando praticamente todo o litoral (101) e o eixo Sul-Sudeste (116).

Porém, volume alto não significa maior risco por acidente. Olhando o índice de gravidade (mortos por acidente):

BR-316 tem o maior índice (0,1685) — quase 3x maior que a BR-101 (0,0576), mesmo tendo 10x menos acidentes

BR-230 (0,1082) e BR-153 (0,0982) também se destacam como proporcionalmente mais letais

Isso sugere uma conclusão importante: BR-101 e BR-116 precisam de atenção por volume, mas BR-316, BR-230 e BR-153 merecem atenção por letalidade por ocorrência — possivelmente por características da via (menos duplicada, mais trechos rurais/sinuosos), algo que podemos cruzar depois com dim_condicao.

Célula 3 — Pergunta 2: Gravidade por fase do dia

In [0]:
df_p2 = (
    fato_acidente
    .join(dim_tempo, on="id_tempo", how="left")
    .groupBy("fase_dia")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .orderBy(F.desc("indice_gravidade"))
)

display(df_p2)

**Interpretação — Resposta à Pergunta 2**

Há uma relação clara: existe sim relação entre horário/fase do dia e gravidade.

"Pleno dia" concentra o maior volume de acidentes (117.982, ~55% do total) — esperado, já que é quando há mais tráfego — mas é a fase com menor gravidade proporcional (0,0594 mortos/acidente).

"Amanhecer" tem o maior índice de gravidade (0,1363) — mais que o dobro do "Pleno dia" — apesar de ter bem menos acidentes em volume absoluto (10.250). Isso é um padrão bem documentado na literatura de segurança viária: acidentes de madrugada/amanhecer tendem a ser mais graves por fatores como sonolência, velocidade mais alta (menos tráfego) e menor visibilidade.

"Plena Noite" também tem gravidade alta (0,116), reforçando a hipótese de que o período noturno/madrugada é desproporcionalmente mais perigoso, mesmo com menos acidentes.

Conclusão: volume de acidentes e gravidade seguem padrões opostos ao longo do dia — políticas de segurança poderiam priorizar fiscalização/iluminação em horários de baixo movimento mas alto risco (madrugada/amanhecer), não só nos horários de pico.

Pergunta 3: "Quais são as principais causas de acidentes e como se relacionam com o número de vítimas?"

In [0]:
df_p3 = (
    fato_acidente
    .join(dim_causa, on="id_causa", how="left")
    .groupBy("causa_acidente")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves"),
        F.sum("feridos_leves").alias("total_feridos_leves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .orderBy(F.desc("qtd_acidentes"))
)

display(df_p3.limit(15))

**Interpretação — Resposta à Pergunta 3**

As causas mais frequentes ("Reação tardia", "Ausência de reação", "Acessar via sem observar") têm gravidade relativamente moderada (0,058-0,073), mas volume altíssimo — juntas somam quase 40% dos acidentes e são as maiores responsáveis pelo total absoluto de vítimas.

Já as causas mais raras em volume, mas extremamente mais letais:

-"Transitar na contramão" — índice de gravidade de 0,3758, mais de 6x maior que a média das causas mais comuns. Só 7.300 acidentes, mas 2.743 mortos — quase tantos mortos quanto a BR-101 inteira (que tem 5x mais acidentes)

-"Ultrapassagem Indevida" — 0,2259, quase 4x mais letal que a média

Conclusão de negócio: fiscalização de volume (reação do condutor, distração) previne o maior número absoluto de acidentes, mas ações contra direção na contramão e ultrapassagens indevidas têm potencial de salvar desproporcionalmente mais vidas por intervenção, mesmo afetando menos ocorrências.

Pergunta 4: "Há diferença nos padrões de acidentes entre dias de semana e finais de semana?"

In [0]:
df_p4 = (
    fato_acidente
    .join(dim_tempo, on="id_tempo", how="left")
    .groupBy("flag_fim_de_semana")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
)

# Normalizando por número de dias (2 dias de FDS vs 5 dias de semana, por semana)
display(df_p4)

In [0]:
df_dias_distintos = (
    dim_tempo
    .select("data_inversa", "flag_fim_de_semana")
    .distinct()
    .groupBy("flag_fim_de_semana")
    .agg(F.count("data_inversa").alias("qtd_dias"))
)

df_p4_normalizado = df_p4.join(df_dias_distintos, on="flag_fim_de_semana", how="left") \
    .withColumn("media_acidentes_por_dia", F.round(F.col("qtd_acidentes") / F.col("qtd_dias"), 2))

display(df_p4_normalizado)

**Interpretação — Resposta à Pergunta 4**

Sim, existe diferença clara nos dois indicadores:

Volume normalizado por dia: finais de semana têm ~19% mais acidentes por dia (219,73 vs 184,77) do que dias úteis — mesmo com menos tráfego de trabalho, provavelmente por viagens de lazer/turismo em rodovias federais.

Gravidade: finais de semana também são ~31% mais letais por acidente (índice 0,0994 vs 0,076) — coerente com fatores já conhecidos como maior consumo de álcool, velocidades mais altas em viagens de lazer, e maior cansaço em deslocamentos longos.

Conclusão: finais de semana são desproporcionalmente mais perigosos tanto em frequência quanto em gravidade — um padrão relevante para políticas de fiscalização reforçada às sextas/sábados/domingos.

Pergunta 5 : "Existe relação entre a quantidade de veículos envolvidos e a gravidade do acidente?"

In [0]:
df_p5 = (
    fato_acidente
    .groupBy("veiculos")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .filter(F.col("qtd_acidentes") >= 30)  # remove categorias muito raras (ruído estatístico)
    .orderBy("veiculos")
)

display(df_p5)

**Interpretação — Resposta à Pergunta 5**

Sim, existe uma relação clara e quase monotônica: quanto mais veículos envolvidos, maior a gravidade do acidente.

Acidentes com 1 veículo (saída de pista, capotamento sozinho) têm o menor índice (0,0391)

Acidentes com 2 veículos já dobram a gravidade (0,083)

A partir de 5+ veículos, o índice ultrapassa 0,20 — mais de 5x a gravidade de um acidente com 1 veículo só

O pico em 11 veículos (0,4211) é estatisticamente mais instável (apenas 57 casos), mas a tendência geral é muito consistente até ali

Conclusão: colisões múltiplas (engavetamentos) são desproporcionalmente mais letais que acidentes isolados, reforçando a importância de distância segura entre veículos (aliás, uma das causas mais frequentes que já vimos na Pergunta 3).